In [3]:
# exracting checkable factual claims from the retracted paper abstracts
# importing the necessary packages

import os
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# loading the environmen variables 
load_dotenv()

# setting the API key
api_key = os.getenv('OPENAI_API_KEY')
print('key loaded:', (api_key[:7] + '...') if api_key else 'not found')

# setting up the OpenAI connection
client = OpenAI(api_key = api_key)

key loaded: sk-proj...


In [4]:
# setting up the function to call the LLM - can be used for any LLM

# sending a prompt to the LLM and returning a text response
# temperature set to 0 to make the output deterministic (needed for reproducibility)
def call_llm(prompt, model = 'gpt-4o-mini', temperature = 0):
    response = client.chat.completions.create(
        model = model,
        temperature = temperature,
        messages = [{'role': 'user', 'content': prompt}],)

    return response.choices[0].message.content.strip()

In [5]:
# the promp's two jobs are to 1) extract the paper's main finding as one factual claim
# and 2) judge whether the abstract actually contains a claim
# setting to a JSON reponse for more reliable parsing

EXTRACTION_PROMPT = '''You are helping build a fact-verification benchmark from biomedical and life sciences paper abstracts.

Given the abstract: 
1. Decide whether it states a single, specific, and checkable factual finding (measureable effect, relationship or quantity).
Qualitative/interpretive studies, pure methodology descriptions, or reviews do NOT count.
2. If yes, extract the finding as ONE concise, self-contained declarative claim (one sentence, no citations, understandable without the abstract).

Respond with JSON and copy this format exactly:
{{"checkable": true or false, "claim": "the one-sentence claim or empty string if not checkable"}}
Do not add anything else to the response.

Abstract: {abstract}'''

In [6]:
# necessary for the abstract responses
import json

# designing the function to extract the claims
def extract_claim(abstract):

    # runs one abstract through the LLM to return checkable: bool and claim: str
    # returns False if anything fails
    prompt = EXTRACTION_PROMPT.format(abstract = abstract)
    try: 
        raw = call_llm(prompt)

        # stripping potential JSON wrap formatting before parsing
        raw = raw.replace('```json', '').replace('```', '').strip()
        data = json.loads(raw)
        return bool(data.get('checkable', False)), data.get('claim', '').strip()
    
    # displays any errors that occur
    except Exception as e:
        print('parsing/LLM error:', e)
        return False, ''

In [7]:
# loading the cleaned data pool
pool = pd.read_csv('../data/interim/tier1_pool.csv', dtype = str, keep_default_na = False)
print('full data pool:', len(pool))

# small sample of the papers that will be used to build the checkable pairs
SAMPLE_SIZE = 20
sample = pool.sample(SAMPLE_SIZE, random_state = 42).copy()

print('sample:', len(sample))

full data pool: 14984
sample: 20


In [8]:
# grabbing the abstracts from NCBI
from Bio import Entrez
import time
Entrez.email = 'kere00003@uni-saarland.de'   
Entrez.api_key = None 

# function to get abstracts
def fetch_abstract(pmid): 

    # get the abstract for a PubMed ID, if there is none, try
    try: 
        h = Entrez.efetch(db = 'pubmed', id = str(pmid), rettype = 'abstract', retmode = 'xml')
        rec = Entrez.read(h); h.close()
        art = rec['PubmedArticle'][0]['MedlineCitation']['Article']
        parts = art.get('Abstract', {}).get('AbstractText', [])
        return " ".join(str(p) for p in parts).strip()
    except Exception:
        return ''

# fetcihng an abstract for each paper in the sample pool 
abstracts = []
for p in sample['OriginalPaperPubMedID']:
    abstracts.append(fetch_abstract(p))

    # setting retrieval rate
    time.sleep(0.34)   
sample['abstract'] = abstracts

# dropping papers with no usable abstract
sample = sample[sample['abstract'].str.split().str.len() > 20].copy()

print('papers with usable abstracts:', len(sample))

papers with usable abstracts: 20


In [9]:
# running each sampled abstract through the LLM to extract a checkable claim

# collecting one dictionary per paper
results = []

# looping over each paper in the sample (iterrows gives index, row — we ignore the index with _)
for _, row in sample.iterrows():

    # runs the abstract through extract_claim and returns (checkable: bool, claim: str)
    checkable, claim = extract_claim(row['abstract'])

    # storing the given paper's information + the extraction result
    results.append({

        # allows us to trace back to the original paper
        'pmid': row['OriginalPaperPubMedID'],

        'title': row['Title'],

        # retraction reason for later analysis
        'reason': row['Reason'],
        'abstract': row['abstract'],

        # did the LLM find a checkable claim?
        'checkable': checkable,

        # returns the extracted claim or an empty result
        'claim': claim,
    })
    # setting a pause between API calls to control the rate 
    time.sleep(0.2)

# turning the list of dicts into a dataframe for easier inspection
claims_df = pd.DataFrame(results)

# printing the results
print('total processed:', len(claims_df))
print('checkable claims:', claims_df['checkable'].sum())

total processed: 20
checkable claims: 19


In [10]:
# saving intermediate results 
claims_df.to_csv('../data/interim/claims_stage1.csv', index = False)

In [11]:
# checking the results to see whether the claims are good
for _, r in claims_df[claims_df['checkable']].iterrows():
    print('claim:', r['claim'])
    print('from:', r['title'][:90])
    print('abstract start:', r['abstract'][:200], '...')

claim: miR-140 inhibits the tumor formation and metastasis of colorectal cancer in vivo.
from: MicroRNA-140 Inhibits the Epithelial-Mesenchymal Transition and Metastasis in Colorectal C
abstract start: MicroRNA-140, a cartilage-specific microRNA, has recently been implicated in the cancer progression. However, the comprehensive role of miR-140 in the invasion and metastasis of colorectal cancer (CRC ...
claim: Microsurgical strabismus correction in children results in short operation time, less intraoperative blood loss, short hospital stay, high efficiency, and fewer complications.
from: Observation on the Curative Effect of Microsurgery in 154 Children with Strabismus and Ana
abstract start: Strabismus is a common ophthalmic disease in the process of child body development, in which the two eyes cannot gaze at the target at the same time, and the incidence of this disease of children is h ...
claim: A higher MCA-PSV (MoM 1.7 compared to 1.5 MoM) can accurately predict the recurrence 

In [12]:
# sanity check for control 
# loading full Retraction Watch database
# collecting every paper's PubMed ID into a set for quicker lookup

retraction_full = pd.read_csv('../data/raw/retraction_watch.csv', dtype = str, keep_default_na = False)

# collecting all useable original paper PubMed IDs
retracted_pmids = set(p.strip() for p in retraction_full['OriginalPaperPubMedID']
                      if p.strip().isdigit() and p.strip() != '0')

print('total retracted PMIDs to exclude:', len(retracted_pmids))

total retracted PMIDs to exclude: 31841


In [13]:
# prompting an LLM to search PubMed articles for relevant terms rather than a keyword match

QUERY_PROMPT = '''Turn this biomedical claim into a SHORT PubMed search query.
Output ONLY 2-4 key terms: the main entity (gene/drug/method) and the main condition or outcome.
Drop specific numbers, cell-line names and secondary details. Ensure the query is broad enough to find related papers.
Do not add anything else (no field tags, no quotes, no explanation, etc).

Claim: {claim}'''

# creating the function for the query 
def build_query(claim): 

    # asking the LLM for relevant PubMed search terms for a given claim
    try: 
        return call_llm(QUERY_PROMPT.format(claim = claim)).strip()

    # in case of errors 
    except Exception as e:
        print('error:', e)
        return ''

In [14]:
from Bio import Entrez

# function to search PubMed
def search_pubmed(query, exclude_pmid, max_results = 5):

    # searchihg PubMed for a query, returning potential PMIDs and excluding retractions and its associated paper
    try: 
        h = Entrez.esearch(db = 'pubmed', term = query, retmax = max_results * 3)
        rec = Entrez.read(h); h.close()
        hits = rec.get('IdList', [])

    # error handling
    except Exception as e:
        print('  search error:', e)
        return []

    # keeping only non-retracted papers
    clean = [pmid for pmid in hits
             if pmid not in retracted_pmids and str(pmid) != str(exclude_pmid)]

    return clean[:max_results]

In [15]:
# for each checkable claim, this cell builds the query, searches PubMed and finds the non-retraced papers

checkable = claims_df[claims_df['checkable']].copy()

retrieval_results = []
for _, row in checkable.iterrows():
    query = build_query(row['claim'])
    candidates = search_pubmed(query, exclude_pmid = row['pmid'])
    retrieval_results.append({
        'pmid': row['pmid'],
        'claim': row['claim'],
        'query': query,
        'candidate_pmids': candidates,
        'n_candidates': len(candidates),})

    # set to control PubMed retrieval rate
    time.sleep(0.34) 

retrieval_df = pd.DataFrame(retrieval_results)

# calculations for viable papers
found = (retrieval_df['n_candidates'] > 0).sum()
print(f'claims with >= 1 candidate: {found} / {len(retrieval_df)}')
print(f'median candidates per claim: {int(retrieval_df["n_candidates"].median())}')

claims with >= 1 candidate: 14 / 19
median candidates per claim: 5


In [16]:
# inspecting the query results
for _, r in retrieval_df.iterrows():
    print(f"[{r['n_candidates']} candidates] query: {r['query']}")
    print(f"  claim: {r['claim'][:90]}")
    print('-' * 90)

[5 candidates] query: miR-140 colorectal cancer
  claim: miR-140 inhibits the tumor formation and metastasis of colorectal cancer in vivo.
------------------------------------------------------------------------------------------
[5 candidates] query: strabismus surgery children outcomes
  claim: Microsurgical strabismus correction in children results in short operation time, less intr
------------------------------------------------------------------------------------------
[5 candidates] query: MCA-PSV fetal anemia transfusions
  claim: A higher MCA-PSV (MoM 1.7 compared to 1.5 MoM) can accurately predict the recurrence of se
------------------------------------------------------------------------------------------
[0 candidates] query: sCLU recurrence-free survival
  claim: Elevated sCLU was significantly associated with recurrence-free survival in groups 1 and 3
------------------------------------------------------------------------------------------
[5 candidates] query: nursing 

In [17]:
# getting the abstract text for every usable PMID; this is later used to score entailment against a claim
# retrieval_df has a list of usable PMIDs per claim

def fetch_candidates(pmid_list):

    # getting abstracts and returning the list of {pmid, abstract} dictionaries
    # skips those with no usable abstract
    out = []
    for pmid in pmid_list:
        abstract = fetch_abstract(pmid)
        time.sleep(0.34) 

        # keeping only usable abstracts
        if len(abstract.split()) > 20:   
            out.append({'pmid': pmid, 'abstract': abstract})
    return out

# fetching candidates for each claim
retrieval_df['candidates'] = retrieval_df['candidate_pmids'].apply(fetch_candidates)

# reporting how many claims still have >= 1 usable candidate after abstract fetching
retrieval_df['n_usable'] = retrieval_df['candidates'].apply(len)
print('claims with >=1 usable candidate abstract:', (retrieval_df['n_usable'] > 0).sum(), '/', len(retrieval_df))

claims with >=1 usable candidate abstract: 14 / 19


In [18]:
retrieval_df.to_pickle('../data/interim/retrieval_stage2.pkl')

In [ ]:
# setting up the scientific NLI model 

# neural network packages
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# SciFact-fine-tuned DeBERTa: given a claim + evidence, classifies support / contradict / no-evidence
NLI_MODEL = 'MilosKosRad/TextualEntailment_DeBERTa_preprocessedSciFACT'

tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL)

# no training
model.eval()

# printing the label mappings
print('label mapping:', model.config.id2label)

c:\Users\kezia\Documents\retraction-aware-citeeval\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

label mapping: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2'}


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.74GB            

model.safetensors: downloading bytes:           |  0.00B            

In [20]:
import torch

def raw_scores(premise, hypothesis):

    # returning the model's probability across the 3 labels 
    inputs = tokenizer(premise, hypothesis, return_tensors = 'pt', truncation = True, max_length = 512)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim = -1)[0]
    return probs.tolist()

# test 1 - case where the premise clearly supports the hypothesis
support_case = raw_scores(
    'Aspirin significantly reduces the risk of heart attack in adults.',
    'Aspirin lowers heart attack risk.'
)
print('support case:', [round(x, 3) for x in support_case])

# test 2 - premise clearly contradicts the hypothesis
contradict_case = raw_scores(
    'Aspirin significantly reduces the risk of heart attack in adults.',
    'Aspirin increases heart attack risk.'
)
print("conrtradiction case:", [round(x, 3) for x in contradict_case])

# test 3 - premise is unrelated to hypothesis
neutral_case = raw_scores(
    'The Eiffel Tower is located in Paris, France.',
    'Aspirin lowers heart attack risk.'
)
print('neutral case:', [round(x, 3) for x in neutral_case])

support case: [0.117, 0.857, 0.027]
conrtradiction case: [0.02, 0.008, 0.972]
neutral case: [0.995, 0.004, 0.001]


In [21]:
# scores how strongly a passage supports a claim using the NLI model
# premise = the evidence passage (abstract), hypothesis = the claim.

SUPPORT_INDEX = 1

# function t ocheck support
def support_score(passage, claim):

    # returning the probability that the modek's passage supports its claim (0 to 1)
    inputs = tokenizer(passage, claim, return_tensors = 'pt', truncation = True, max_length = 512)
    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim = -1)[0]
    return probs[SUPPORT_INDEX].item()

In [22]:
# scoring how well the retracted abstract supports each claim

# getting claims with usable candidates
gated = retrieval_df[retrieval_df['n_usable'] > 0].copy()

# getting retracted paper's abstract
pmid_to_abstract = dict(zip(claims_df['pmid'], claims_df['abstract']))

gate_results = []
for _, row in gated.iterrows():
    claim = row['claim']
    retracted_abstract = pmid_to_abstract.get(row['pmid'], '')

    # checking how well the retracted paper supports its own claim
    s_retracted = support_score(retracted_abstract, claim)

    # scoring each candidate and keeping the best supported one
    best_valid = None
    best_score = -1.0
    for cand in row['candidates']:
        s = support_score(cand['abstract'], claim)
        if s > best_score:
            best_score = s
            best_valid = cand['pmid']

    gate_results.append({
        'pmid': row['pmid'],
        'claim': claim,

        # retracted abstract score
        's_retracted': s_retracted,       
        'best_valid_pmid': best_valid,    
        's_valid': best_score,            
        'gap': abs(s_retracted - best_score),})

gate_df = pd.DataFrame(gate_results)

# showing the score distribution to pick the threshold (T) and margin (M)
print(gate_df[['s_retracted', 's_valid', 'gap']].describe())
print()
for _, r in gate_df.iterrows():
    print(f'retracted = {r['s_retracted']:.3f}  valid = {r['s_valid']:.3f}  gap = {r['gap']:.3f}  | {r['claim'][:70]}')

       s_retracted    s_valid        gap
count    14.000000  14.000000  14.000000
mean      0.846491   0.920598   0.190996
std       0.248811   0.248462   0.277638
min       0.107888   0.069379   0.000034
25%       0.760415   0.996500   0.004538
50%       0.987354   0.998130   0.066011
75%       0.997541   0.999401   0.262044
max       0.999723   0.999673   0.891476

retracted = 0.996  valid = 0.996  gap = 0.000  | miR-140 inhibits the tumor formation and metastasis of colorectal canc
retracted = 0.879  valid = 0.998  gap = 0.120  | Microsurgical strabismus correction in children results in short opera
retracted = 0.998  valid = 0.999  gap = 0.001  | A higher MCA-PSV (MoM 1.7 compared to 1.5 MoM) can accurately predict 
retracted = 0.999  valid = 0.843  gap = 0.156  | Stratified nursing management significantly improves nursing quality a
retracted = 0.729  valid = 0.069  gap = 0.660  | Irisin treatment significantly increases ATP production and reduces LP
retracted = 0.701  valid = 0.9

In [25]:
# setting up the date to keep a pair only if both the retracted and valid abstracts clearly support the claim

# both abstracts must score at leats 70%
T = 0.7   

# scores must be within this margin of each other
M = 0.2   

# defining the gate function
def passes_gate(row, t = T, m = M):
    return (row['s_retracted'] >= t) and (row['s_valid'] >= t) and (row['gap'] <= m)

gate_df['passes'] = gate_df.apply(passes_gate, axis = 1)

n_pass = gate_df['passes'].sum()
print(f'pairs passing gate (T = {T}, M = {M}): {n_pass} / {len(gate_df)}\n')


for _, r in gate_df[gate_df['passes']].iterrows():
    print(f'retracted = {r['s_retracted']:.3f} valid = {r['s_valid']:.3f} gap = {r['gap']:.3f} | {r['claim'][:65]}')

# getting retraction reason
for _, r in gate_df[~gate_df['passes']].iterrows():
    reasons = []
    if r['s_retracted'] < T: reasons.append(f'retracted<{T}')
    if r['s_valid'] < T:     reasons.append(f'valid<{T}')
    if r['gap'] > M:         reasons.append(f'gap>{M}')
    print(f'[{', '.join(reasons)}] retracted = {r['s_retracted']:.3f} valid = {r['s_valid']:.3f} | {r['claim'][:60]}')

pairs passing gate (T = 0.7, M = 0.2): 10 / 14

retracted = 0.996 valid = 0.996 gap = 0.000 | miR-140 inhibits the tumor formation and metastasis of colorectal
retracted = 0.879 valid = 0.998 gap = 0.120 | Microsurgical strabismus correction in children results in short 
retracted = 0.998 valid = 0.999 gap = 0.001 | A higher MCA-PSV (MoM 1.7 compared to 1.5 MoM) can accurately pre
retracted = 0.999 valid = 0.843 gap = 0.156 | Stratified nursing management significantly improves nursing qual
retracted = 0.854 valid = 0.997 gap = 0.143 | Ultrasound-guided high-intensity focused ultrasound improves the 
retracted = 0.987 valid = 0.999 gap = 0.011 | Administration of a retroviral vector containing ACE antisense ca
retracted = 0.988 valid = 1.000 gap = 0.012 | The ADC value of the measured patient's osteomyelitis area was si
retracted = 0.987 valid = 1.000 gap = 0.012 | ARPE19, ARPE19mel, and hfRPE cells failed to form a tight barrier
retracted = 0.999 valid = 0.997 gap = 0.002 | Drainage a

In [ ]:
# sanity check - false negative

# getting the lowest-scoring retracted case
odd = gate_df[gate_df['s_retracted'] < 0.3].iloc[0]  
print("claim:", odd['claim'])
print('\ns_retracted:', round(odd['s_retracted'], 3), 's_valid:', round(odd['s_valid'], 3))
print('\nRETRACTED ABSTRACT:')
print(pmid_to_abstract.get(odd['pmid'], '')[:800])

CLAIM: Cabozantinib drives colorectal cancer cell apoptosis via a PUMA-dependent mechanism.

s_retracted: 0.048  s_valid: 0.982

RETRACTED ABSTRACT:
Cabozantinib is a multi-kinase inhibitor targeting MET, AXL, and VEGFR2, and has been approved for use in multiple malignancies. The means by which Cabozantinib acts to target colorectal cancer (CRC) cells remains poorly understood, and we sought to investigate how this drug disrupts cell growth in CRC cells and how it interacts to enhance the efficacy of other chemotherapeutic agents. In this study, we found that Cabozantinib activated a p65-dependent signaling pathway in response to both inhibition of AKT and activation of glycogen synthase kinase 3β (GSK3β), leading to upregulation of PUMA in CRC cells regardless of p53 activity. PUMA upregulation facilitates CRC apoptosis in response to Cabozantinib, which also acts synergistically with the chemotherapeutic agents Cetuximab and 5-FU to
